# Edge IIoT - Binary Classification


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [23]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATASETS

# Seleccionamos el dataset a analizar
dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

print("\n--- Dataset Information ---")
print(f"Name: {dataset_name.upper()}")
print(f"Path: {config['processed_path']}\n")

filename = "ML-EdgeIIoT-dataset"

csv_path = config['processed_path'] / f"{filename}.pkl"
print(f"CSV Path: {csv_path}\n")

print("Loading dataset... (This may take a while)")
df = pd.read_pickle(csv_path)

print(f"\nDataset loaded with shape: {df.shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

--- Dataset Information ---
Name: EDGE_IIOT
Path: E:\ML-NIDS-IIoT-tmp\data\processed\edge_iiot

CSV Path: E:\ML-NIDS-IIoT-tmp\data\processed\edge_iiot\ML-EdgeIIoT-dataset.pkl

Loading dataset... (This may take a while)

Dataset loaded with shape: (155027, 59)


## 1. Pre-processing

In [24]:
df_naive = df.select_dtypes(include=['number'])

target = 'Attack_label'
X = df_naive.drop(columns=[target])
y = df_naive[target]

Y_str = df['Attack_type'] # Used to stratify later

In [25]:
categorical_cols = X.select_dtypes(include=['str', 'object', 'category']).columns
print(f"Check for categorical columns: {list(categorical_cols)}", "\t(Should only include 'Attack_type')")
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

Check for categorical columns: [] 	(Should only include 'Attack_type')


In [26]:
X.columns

Index(['icmp.seq_le', 'http.content_length', 'http.response', 'http.tls_port',
       'tcp.ack', 'tcp.len', 'tcp.seq', 'udp.stream', 'udp.time_delta',
       'dns.retransmission', 'dns.retransmit_request',
       'dns.retransmit_request_in', 'mqtt.conack.flags',
       'mqtt.conflag.cleansess', 'mqtt.len', 'mqtt.msgtype', 'mbtcp.len',
       'mbtcp.trans_id', 'mbtcp.unit_id', 'frame.time.order',
       'frame.time.delta', 'ip.src_category_Malformed',
       'ip.src_category_Private', 'ip.src_category_Public',
       'ip.src_category_Reserved', 'ip.dst_category_Malformed',
       'ip.dst_category_Private', 'ip.dst_category_Public',
       'ip.dst_category_Reserved', 'arp.opcode_request', 'arp.opcode_reply',
       'http.request.method_get', 'http.request.method_post',
       'http.request.method_options', 'http.request.method_trace',
       'tcp.nullchecksum', 'tcp.flag.res', 'tcp.flag.ns', 'tcp.flag.cwr',
       'tcp.flag.ece', 'tcp.flag.urg', 'tcp.flag.ack', 'tcp.flag.psh',
       'tc

In [27]:
cols_to_drop = ['frame.time.order', 'frame.time.delta']

for col in X.columns:
    if col.startswith('ip.src_category') or col.startswith('ip.dst_category'):
        cols_to_drop.append(col)

X = X.drop(columns=cols_to_drop)

In [28]:
print(f"\nFinal feature set shape after encoding: {X.shape}")
print(f"NaN values in target variable: {y.isna().sum()}")
print(f"NaN values in features: {X.isna().sum().sum()}")


Final feature set shape after encoding: (155027, 40)
NaN values in target variable: 0
NaN values in features: 0


In [29]:
X_train, X_test, y_train, y_test, Y_str_train, Y_str_test = train_test_split(
    X, y, Y_str, test_size=0.2, random_state=42, stratify=Y_str
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_results = {}

X_train shape: (124021, 40)
X_test shape: (31006, 40)


## 2. LazyPredict


In [30]:
from lazypredict.Supervised import LazyClassifier

clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None, timeout=300)
models, predictions = clf.fit(X_train_scaled, X_test_scaled, y_train, y_test)

display(models)


Large dataset detected (124021 samples). Training all models may take a long time. Consider using a subset or setting max_models/timeout.


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
Model,,,,,,,
LGBMClassifier,0.975263,0.920863,0.992878,0.974379,0.975967,0.975263,0.628470
XGBClassifier,0.974940,0.920671,0.992894,0.974057,0.975576,0.974940,0.661512
CatBoostClassifier,0.974811,0.920427,0.992882,0.973924,0.975437,0.974811,12.354263
ExtraTreesClassifier,0.975069,0.920328,0.992465,0.974173,0.975776,0.975069,5.137071
RandomForestClassifier,0.967135,0.894862,0.985660,0.965519,0.968368,0.967135,4.706707
BaggingClassifier,0.967039,0.894804,0.985649,0.965423,0.968242,0.967039,1.603342
ExtraTreeClassifier,0.967071,0.894739,0.991471,0.965451,0.968296,0.967071,0.268897
DecisionTreeClassifier,0.967006,0.894701,0.985494,0.965387,0.968212,0.967006,0.377252
KNeighborsClassifier,0.966813,0.894166,0.962979,0.965177,0.968020,0.966813,6.198502
